# EDALens — testing the transformation library

Exercises `utils.data_processing` on branch `fix/transformation-integrity`: every operation, its failure modes (`OperationError`), immutability, history replay, atomic session updates and undo. The last sections launch the Streamlit app with `edalens.show()` and run the pytest suite.

## 1. Setup

In [9]:
import edalens

app = edalens.show()              # open_browser=False to only get the URL
app

<EDALens server http://127.0.0.1:8504/edalens-74b2f69e651a89d1/ (running)>

In [8]:
app.url                          # e.g. http://127.0.0.1:8501/edalens-3f9c.../
app.stop()                       # stops only this server; safe to call twice

with edalens.show(open_browser=False) as app:   # stopped when the block ends
    print(app.url)

http://127.0.0.1:8502/edalens-dde055af9a5c7cf8/


In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import plotly.express as px
from pandas.testing import assert_frame_equal

from edalens.utils.data_processing import (
    OperationError, apply_operation, apply_operations, column_types,
    describe_operation, run_operation, undo_last_operation,
)

print("repo:", REPO)
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)

repo: /home/salva/Documents/claude/deaviz
python 3.13.5 | pandas 2.3.3 | numpy 1.26.4


## 2. Sample dataset
Mixed dtypes with missing values: datetime strings, categorical, int, float, nullable `Int64`, text and bool, plus a non-default index.

In [2]:
rng = np.random.default_rng(0)
n = 60
raw = pd.DataFrame(
    {
        "date": pd.date_range("2024-01-01", periods=n, freq="13h").strftime("%Y-%m-%d %H:%M"),
        "city": rng.choice(["Quito", "Lima", "Bogota"], n),
        "temp": rng.normal(20, 4, n).round(1),
        "humidity": rng.integers(30, 90, n),
        "visits": pd.array(rng.integers(0, 10, n), dtype="Int64"),
        "note": rng.choice(["ok", "check", "late"], n),
        "flag": rng.choice([True, False], n),
    },
    index=pd.RangeIndex(100, 100 + n),
)
# Inject missing values
raw.loc[raw.sample(frac=0.15, random_state=1).index, "temp"] = np.nan
raw.loc[raw.sample(frac=0.10, random_state=2).index, "visits"] = pd.NA
raw.loc[raw.sample(frac=0.10, random_state=3).index, "note"] = None
raw["city"] = raw["city"].astype("category")

display(raw.head())
print(raw.dtypes.to_string())
print("\nmissing per column:\n", raw.isna().sum().to_string())

,date,city,temp,humidity,visits,note,flag
100,2024-01-01 00:00,Bogota,16.0,82,<NA>,ok,False
101,2024-01-01 13:00,Lima,19.2,30,9,ok,True
102,2024-01-02 02:00,Lima,NaN,37,1,ok,True
103,2024-01-02 15:00,Quito,22.2,81,5,check,False
104,2024-01-03 04:00,Quito,20.9,34,7,check,True


date          object
city        category
temp         float64
humidity       int64
visits         Int64
note          object
flag            bool

missing per column:
 date        0
city        0
temp        9
humidity    0
visits      6
note        6
flag        0


In [3]:
def check_unchanged(before, after_op_input):
    """Every operation must return a new frame and never mutate its input."""
    assert_frame_equal(before, after_op_input)

snapshot = raw.copy(deep=True)

## 3. Individual operations

### 3.1 `select_columns` — keeps the frame's own column order

In [4]:
op = {"type": "select_columns", "columns": ["temp", "date", "city"]}
out = apply_operation(raw, op)
print(describe_operation(op))
assert list(out.columns) == ["date", "city", "temp"]   # original order, not request order
assert out.index.equals(raw.index)
check_unchanged(snapshot, raw)
out.head()

Selected columns: temp, date, city


,date,city,temp
100,2024-01-01 00:00,Bogota,16.0
101,2024-01-01 13:00,Lima,19.2
102,2024-01-02 02:00,Lima,NaN
103,2024-01-02 15:00,Quito,22.2
104,2024-01-03 04:00,Quito,20.9


### 3.2 `cast_types`

In [5]:
op = {"type": "cast_types", "numeric_cols": ["humidity"], "categorical_cols": ["flag", "visits"]}
out = apply_operation(raw, op)
print(describe_operation(op))
print(out.dtypes.to_string())
# Missing values stay missing (not the string 'nan' / '<NA>')
assert out["visits"].isna().sum() == raw["visits"].isna().sum()
assert not out["visits"].isin(["nan", "<NA>", "None"]).any()
check_unchanged(snapshot, raw)
print("numeric/categorical:", column_types(out))

Cast types - Numeric: humidity, Categorical: flag, visits
date          object
city        category
temp         float64
humidity       int64
visits        object
note          object
flag          object
numeric/categorical: (['temp', 'humidity'], ['date', 'city', 'visits', 'note', 'flag'])


### 3.3 `subsample` — deterministic (seed 42), original row order kept

In [6]:
op = {"type": "subsample", "sample_pct": 25}
a, b = apply_operation(raw, op), apply_operation(raw, op)
assert_frame_equal(a, b)
assert len(a) == round(len(raw) * 0.25)
assert a.index.is_monotonic_increasing
print(describe_operation(op), "->", len(a), "rows; index:", list(a.index[:8]), "...")
assert_frame_equal(apply_operation(raw, {"type": "subsample", "sample_pct": 100}), raw)

Subsampled to 25% of data -> 15 rows; index: [100, 103, 105, 112, 113, 131, 133, 136] ...


### 3.4 `melt`

In [7]:
op = {"type": "melt", "id_vars": ["date", "city"]}
small = apply_operation(raw, {"type": "select_columns", "columns": ["date", "city", "temp", "humidity"]})
out = apply_operation(small, op)
print(describe_operation(op), out.shape)
assert len(out) == len(small) * 2 and list(out.columns) == ["date", "city", "variable", "value"]
out.head()

Melted with ID vars: date, city (120, 4)


,date,city,variable,value
0,2024-01-01 00:00,Bogota,temp,16.0
1,2024-01-01 13:00,Lima,temp,19.2
2,2024-01-02 02:00,Lima,temp,NaN
3,2024-01-02 15:00,Quito,temp,22.2
4,2024-01-03 04:00,Quito,temp,20.9


### 3.5 `date_features`

In [8]:
op = {"type": "date_features", "date_column": "date", "dayfirst": False}
out = apply_operation(raw, op)
print(describe_operation(op))
display(out[["date", "Year", "Month", "Day", "DayOfWeek", "Hour"]].head())
assert out["Year"].dtype == "int64"           # no missing dates -> int
# With a missing / unparseable date the features become float (NaN-capable)
messy = raw.copy(); messy.loc[messy.index[0], "date"] = "not a date"
out2 = apply_operation(messy, op)
assert out2["Year"].dtype == "float64" and np.isnan(out2.loc[messy.index[0], "Year"])

Extracted date features from date (dayfirst: False)


,date,Year,Month,Day,DayOfWeek,Hour
100,2024-01-01 00:00:00,2024,1,1,0,0
101,2024-01-01 13:00:00,2024,1,1,0,13
102,2024-01-02 02:00:00,2024,1,2,1,2
103,2024-01-02 15:00:00,2024,1,2,1,15
104,2024-01-03 04:00:00,2024,1,3,2,4


/home/salva/Documents/claude/deaviz/utils/data_processing.py:149: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(source, errors="coerce", dayfirst=operation["dayfirst"])


### 3.6 `remove_nan`

In [9]:
out = apply_operation(raw, {"type": "remove_nan"})
print(len(raw), "->", len(out), "rows")
assert out.notna().all().all()
check_unchanged(snapshot, raw)

60 -> 41 rows


### 3.7 `impute` — KNN for numeric, mode for categorical/text/bool
Columns, index and order are preserved; integer columns that receive imputed values become float; unchanged columns keep their dtype.

In [10]:
out = apply_operation(raw, {"type": "impute"})
cmp = pd.DataFrame({"dtype before": raw.dtypes.astype(str), "dtype after": out.dtypes.astype(str),
                    "NaN before": raw.isna().sum(), "NaN after": out.isna().sum()})
display(cmp)
assert list(out.columns) == list(raw.columns) and out.index.equals(raw.index)
assert out[["temp", "visits", "note", "city"]].notna().all().all()
assert out["visits"].dtype == "Float64"             # nullable Int64 -> Float64
for col in ["humidity", "flag", "date"]:            # untouched columns keep dtype & values
    pd.testing.assert_series_equal(out[col], raw[col])
check_unchanged(snapshot, raw)

,dtype before,dtype after,NaN before,NaN after
date,object,object,0,0
city,category,category,0,0
temp,float64,float64,9,0
humidity,int64,int64,0,0
visits,Int64,Float64,6,0
note,object,object,6,0
flag,bool,bool,0,0


In [11]:
mask = raw["temp"].isna()
plot_df = out.assign(imputed=np.where(mask, "imputed (KNN)", "observed")).reset_index()
px.scatter(plot_df, x="index", y="temp", color="imputed", symbol="city",
           title="temp: observed vs KNN-imputed values")

## 4. Failure modes → `OperationError` (input never modified)

In [12]:
bad_ops = {
    "unknown type":           (raw, {"type": "explode"}),
    "select nothing":         (raw, {"type": "select_columns", "columns": []}),
    "select missing column":  (raw, {"type": "select_columns", "columns": ["nope"]}),
    "numeric & categorical":  (raw, {"type": "cast_types", "numeric_cols": ["temp"], "categorical_cols": ["temp"]}),
    "text -> numeric":        (raw, {"type": "cast_types", "numeric_cols": ["note"], "categorical_cols": []}),
    "subsample 0%":           (raw, {"type": "subsample", "sample_pct": 0}),
    "subsample 150%":         (raw, {"type": "subsample", "sample_pct": 150}),
    "melt w/o id vars":       (raw, {"type": "melt", "id_vars": []}),
    "melt all id vars":       (raw, {"type": "melt", "id_vars": list(raw.columns)}),
    "unparseable dates":      (raw, {"type": "date_features", "date_column": "note", "dayfirst": False}),
    "date feature collision": (raw.assign(Year=1), {"type": "date_features", "date_column": "date", "dayfirst": False}),
    "remove_nan drops all":   (raw.assign(empty=np.nan), {"type": "remove_nan"}),
    "duplicate columns":      (pd.DataFrame([[1, 2]], columns=["a", "a"]), {"type": "impute"}),
}
rows = []
for name, (df, op) in bad_ops.items():
    before = df.copy(deep=True)
    try:
        apply_operation(df, op)
        rows.append((name, "NO ERROR ❌", ""))
    except OperationError as exc:
        rows.append((name, "OperationError ✅", str(exc)))
    assert_frame_equal(before, df)
res = pd.DataFrame(rows, columns=["case", "result", "message"])
with pd.option_context("display.max_colwidth", None):
    display(res)
assert res["result"].str.contains("✅").all()

/home/salva/Documents/claude/deaviz/utils/data_processing.py:149: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(source, errors="coerce", dayfirst=operation["dayfirst"])


,case,result,message
0,unknown type,OperationError ✅,Unknown operation type: 'explode'.
1,select nothing,OperationError ✅,Select at least one column to keep.
2,select missing column,OperationError ✅,Selected columns refer to columns missing from the current data: nope.
3,numeric & categorical,OperationError ✅,Columns cannot be both numeric and categorical: temp.
4,text -> numeric,OperationError ✅,Column 'note' has no values that can be read as numbers.
5,subsample 0%,OperationError ✅,Sample percentage must be greater than 0 and at most 100.
6,subsample 150%,OperationError ✅,Sample percentage must be greater than 0 and at most 100.
7,melt w/o id vars,OperationError ✅,Select at least one ID variable before melting.
8,melt all id vars,OperationError ✅,Melting needs at least one non-ID column to turn into rows.
9,unparseable dates,OperationError ✅,No values in column 'note' could be parsed as dates.


## 5. Replay with `apply_operations` = step-by-step `apply_operation`

In [13]:
pipeline = [
    {"type": "select_columns", "columns": ["date", "city", "temp", "visits", "note"]},
    {"type": "date_features", "date_column": "date", "dayfirst": False},
    {"type": "impute"},
    {"type": "subsample", "sample_pct": 50},
]
replayed = apply_operations(raw, pipeline)
stepwise = raw
for op in pipeline:
    stepwise = apply_operation(stepwise, op)
assert_frame_equal(replayed, stepwise)
check_unchanged(snapshot, raw)
for i, op in enumerate(pipeline, 1):
    print(i, describe_operation(op))
replayed.head()

1 Selected columns: date, city, temp, visits, note
2 Extracted date features from date (dayfirst: False)
3 Imputed missing values (KNN numeric, mode categorical)
4 Subsampled to 50% of data


,date,city,temp,visits,note,Year,Month,Day,DayOfWeek,Hour
100,2024-01-01 00:00:00,Bogota,16.0,2.6,ok,2024,1,1,0,0
103,2024-01-02 15:00:00,Quito,22.2,5.0,check,2024,1,2,1,15
104,2024-01-03 04:00:00,Quito,20.9,7.0,check,2024,1,3,2,4
105,2024-01-03 17:00:00,Quito,21.4,2.0,ok,2024,1,3,2,17
106,2024-01-04 06:00:00,Quito,17.4,2.0,late,2024,1,4,3,6


In [14]:
# A failing step reports which step failed
try:
    apply_operations(raw, pipeline + [{"type": "melt", "id_vars": ["missing_col"]}])
except OperationError as exc:
    print("OperationError:", exc)

OperationError: Step 5 (Melted with ID vars: missing_col) failed: ID variables refer to columns missing from the current data: missing_col.


## 6. Session state: `run_operation` / `undo_last_operation`
A plain dict stands in for `st.session_state`. Both direct mode and tracking mode must give identical results, and a failed operation must leave state untouched.

In [15]:
def new_state(tracking):
    return {"raw_df": raw.copy(), "filtered_df": raw.copy(), "operations": [],
            "enable_operation_tracking": tracking}

results = {}
for tracking in (False, True):
    st = new_state(tracking)
    for op in pipeline:
        run_operation(st, op)
    results[tracking] = st
    print(f"tracking={tracking}: {len(st['operations'])} ops, shape={st['filtered_df'].shape}, is_imputed={st['is_imputed']}")
assert_frame_equal(results[False]["filtered_df"], results[True]["filtered_df"])
print("direct and tracked modes agree ✅")

tracking=False: 4 ops, shape=(30, 10), is_imputed=True
tracking=True: 4 ops, shape=(30, 10), is_imputed=True
direct and tracked modes agree ✅


In [16]:
import copy

st = new_state(tracking=False)
run_operation(st, pipeline[0])
before = copy.deepcopy({k: v for k, v in st.items()})
try:
    run_operation(st, {"type": "melt", "id_vars": ["does_not_exist"]})
except OperationError as exc:
    print("rejected:", exc)
assert st.keys() == before.keys()
assert st["operations"] == before["operations"]
assert_frame_equal(st["filtered_df"], before["filtered_df"])
print("state unchanged after failure ✅")

rejected: ID variables refer to columns missing from the current data: does_not_exist.
state unchanged after failure ✅


In [17]:
st = new_state(tracking=False)
for op in pipeline:
    run_operation(st, op)
print("ops:", [o["type"] for o in st["operations"]], st["filtered_df"].shape)
undo_last_operation(st)
print("after undo:", [o["type"] for o in st["operations"]], st["filtered_df"].shape)
assert_frame_equal(st["filtered_df"], apply_operations(raw, pipeline[:-1]))
for _ in range(len(st["operations"])):
    undo_last_operation(st)
assert st["operations"] == [] and not st["is_imputed"]
assert_frame_equal(st["filtered_df"], raw)
print("undoing everything restores raw data ✅")
try:
    undo_last_operation(st)
except OperationError as exc:
    print("empty undo:", exc)

ops: ['select_columns', 'date_features', 'impute', 'subsample'] (30, 10)
after undo: ['select_columns', 'date_features', 'impute'] (60, 10)
undoing everything restores raw data ✅
empty undo: There is no operation to undo.


## 7. Launch the Streamlit app from the notebook
`edalens.show()` returns once the server answers its health check. Open the URL to test the UI manually, then run the stop cell.

In [18]:
import edalens

app = edalens.show(open_browser=False)
app

<EDALens server http://127.0.0.1:8501/edalens-fc575a0d7bf2fb69/ (running)>

In [19]:
app.stop()
app

<EDALens server http://127.0.0.1:8501/edalens-fc575a0d7bf2fb69/ (stopped)>

## 8. Run the repository test suite

In [20]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_transformations.py", "tests/test_data_processing.py",
                    "tests/test_views_operations.py"],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])

.............................                                            [100%]
=============================== warnings summary ===============================
tests/test_transformations.py::test_date_features_reject_unparseable_and_colliding_columns
  /home/salva/Documents/claude/deaviz/utils/data_processing.py:149: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
    parsed = pd.to_datetime(source, errors="coerce", dayfirst=operation["dayfirst"])

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
29 passed, 1 warning in 1.41s
 
